DATA IMPORTING

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler,PolynomialFeatures
from sklearn.linear_model import LinearRegression
import pyarrow
import lightgbm as lgb
%matplotlib inline
## Import used libraries and data
BASE_PATH = '/kaggle/input/kaggle/train.parquet'

df = pd.read_parquet(BASE_PATH, engine='pyarrow')
df.head()

,id,code,sub_code,sub_category,horizon,ts_index,feature_a,feature_b,feature_c,feature_d,...,feature_ca,feature_cb,feature_cc,feature_cd,feature_ce,feature_cf,feature_cg,feature_ch,y_target,weight
0,W2MW3G2L__J0G2B0KU__PZ9S1Z4V__25__89,W2MW3G2L,J0G2B0KU,PZ9S1Z4V,25,89,29,16.364093,7.464023,5.966933,...,-0.001686,-0.105328,-0.005045,NaN,-0.133697,2.849819,0.112068,1,-0.551324,40.982572
1,W2MW3G2L__J0G2B0KU__PZ9S1Z4V__1__89,W2MW3G2L,J0G2B0KU,PZ9S1Z4V,1,89,53,2.858806,5.050617,15.906651,...,-0.001686,-0.105328,-0.005045,NaN,-0.133697,2.849819,0.112068,1,-0.315583,150.075406
2,W2MW3G2L__J0G2B0KU__PZ9S1Z4V__3__89,W2MW3G2L,J0G2B0KU,PZ9S1Z4V,3,89,51,9.585452,1.076268,9.004147,...,-0.001686,-0.105328,-0.005045,NaN,-0.133697,2.849819,0.112068,1,-0.362894,115.953552
3,W2MW3G2L__J0G2B0KU__PZ9S1Z4V__10__89,W2MW3G2L,J0G2B0KU,PZ9S1Z4V,10,89,44,8.840588,15.034634,4.170780,...,-0.001686,-0.105328,-0.005045,NaN,-0.133697,2.849819,0.112068,1,-0.667023,64.573073
4,W2MW3G2L__J0G2B0KU__PZ9S1Z4V__25__90,W2MW3G2L,J0G2B0KU,PZ9S1Z4V,25,90,28,2.303825,7.696209,12.896100,...,-0.001622,-0.103809,-0.005135,NaN,-0.174660,2.738606,0.109204,1,-0.437398,41.948761


DATA OVERVIEW

In [2]:
## check the target column
## df.filter(like='target')
##df.info()
##print(df[['code','sub_code','sub_category','horizon','ts_index']].nunique())
##df.describe()
##print(df.dtypes)
## print(df.groupby(['code','sub_code','sub_category']).size())

##print (df["sub_category"].head(5))
##print (df["horizon"].head(10))
##print(df["ts_index"].head(10))
##print(df["ts_index"].min())
##print(df["ts_index"].max())

DATA CLEANING

1. Handle missing data

In [3]:
## We have 86 features columns
fea_cols = [c for c in df.columns if c.startswith('feature_')]
## columns of feature
n_rows = df.shape[0]

null_summary = pd.DataFrame({
    'null_count': df[fea_cols].isnull().sum(),
    'null_ratio': df[fea_cols].isnull().sum() / n_rows
})
low_null_cols = null_summary[
    null_summary['null_ratio'] < 0.05
].index.tolist()
medium_null_cols = null_summary[
    null_summary['null_ratio'] >= 0.05
].index.tolist()
print (len(low_null_cols)) 
## 81
print(len(medium_null_cols)) 
## 5

81
5


For columns with more than 5% missing values, missingness may be non-random, so missing values are retained. Columns with less than 5% missing values are imputed with the median

In [4]:
## Replace columns with less than 5% missing values by the column median
for name_of_col in low_null_cols:
    df[name_of_col] = df[name_of_col].fillna(df[name_of_col].median())

Features with more than 5% missing values were imputed with the median and augmented with missing indicator flags.

In [5]:
for col in medium_null_cols:
    # creat missing flag columns
    df[col + '_missing_flag'] = df[col].isnull().astype(int)
    df[col] = df[col].fillna(df[col].median())

print(df[medium_null_cols].isnull().sum().sum())


0


2. Handling categorical variables

In [6]:
df = pd.get_dummies(df, columns=["sub_category","horizon"], drop_first= False)
df["code"] = df["code"].astype("category")
df["sub_code"] = df["sub_code"].astype("category")
df.head(5)

,id,code,sub_code,ts_index,feature_a,feature_b,feature_c,feature_d,feature_e,feature_f,...,feature_ce_missing_flag,sub_category_DPPUO5X2,sub_category_NQ58FVQM,sub_category_PHHHVYZI,sub_category_PZ9S1Z4V,sub_category_V8BKY1IV,horizon_1,horizon_3,horizon_10,horizon_25
0,W2MW3G2L__J0G2B0KU__PZ9S1Z4V__25__89,W2MW3G2L,J0G2B0KU,89,29,16.364093,7.464023,5.966933,1.622184,10.261360,...,0,False,False,False,True,False,False,False,False,True
1,W2MW3G2L__J0G2B0KU__PZ9S1Z4V__1__89,W2MW3G2L,J0G2B0KU,89,53,2.858806,5.050617,15.906651,10.879453,3.072151,...,0,False,False,False,True,False,True,False,False,False
2,W2MW3G2L__J0G2B0KU__PZ9S1Z4V__3__89,W2MW3G2L,J0G2B0KU,89,51,9.585452,1.076268,9.004147,16.740490,15.166901,...,0,False,False,False,True,False,False,True,False,False
3,W2MW3G2L__J0G2B0KU__PZ9S1Z4V__10__89,W2MW3G2L,J0G2B0KU,89,44,8.840588,15.034634,4.170780,1.584433,5.383462,...,0,False,False,False,True,False,False,False,True,False
4,W2MW3G2L__J0G2B0KU__PZ9S1Z4V__25__90,W2MW3G2L,J0G2B0KU,90,28,2.303825,7.696209,12.896100,13.830051,0.552439,...,0,False,False,False,True,False,False,False,False,True


3. Data formatting

change all features column to float32

In [7]:
df[fea_cols] = df[fea_cols].astype("float32")

In [8]:
df.shape

(5337414, 106)

Due to the large size of the dataset, it was divided for processing.

In [9]:
df = df.sort_values("ts_index")
df_small1 = df[df["ts_index"] <= 720]
df_small2 = df[(df["ts_index"] > 720) & (df["ts_index"] <= 1440)]
df_small3 = df[(df["ts_index"] > 1440) & (df["ts_index"] <= 2160)]
df_small4 = df[(df["ts_index"] > 2160) & (df["ts_index"] <= 2880)]
df_small5 = df[df["ts_index"] > 2880]

4. Time-series validation

In [10]:
X = df_small1.drop(columns =["y_target","weight","id"])
Y = df_small1["y_target"]
w = df_small1["weight"]

Idea is that take top important features of each part of the table

In [11]:
def get_top_features(df_part, top_n = 40):
    X = df_part.drop(columns=["y_target", "weight", "id"])
    Y = df_part["y_target"]
    W = df_part["weight"]

    cut = int(df_part["ts_index"].quantile(0.8))
    train_mask = df_part["ts_index"] <= cut
    valid_mask = df_part["ts_index"] > cut
    X_train = X[train_mask]
    Y_train = Y[train_mask]
    W_train = W[train_mask]

    model = lgb.LGBMRegressor(
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=31,
        min_child_samples=20,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        Y_train,
        sample_weight=W_train
    )

    importance = pd.Series(
        model.booster_.feature_importance(importance_type="gain"),
        index=X_train.columns
    ).sort_values(ascending=False)

    top_features = importance[importance > 0].head(top_n).index.tolist()

    return top_features

In [12]:
top1 = get_top_features(df_small1)
top2 = get_top_features(df_small2)
top3 = get_top_features(df_small3)
top4 = get_top_features(df_small4)
top5 = get_top_features(df_small5)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.371576 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 21981
[LightGBM] [Info] Number of data points in the train set: 613618, number of used features: 103
[LightGBM] [Info] Start training from score -0.000081
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.485068 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 21981
[LightGBM] [Info] Number of data points in the train set: 794905, number of used features: 103
[LightGBM] [Info] Start training from score -0.000096
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.567069 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 21986
[LightGBM] [Info] Number of data points in the train set: 895674, number of used features: 103
[LightGBM] [Info

In [15]:
from collections import Counter
all_features = top1 + top2 + top3 + top4 + top5

feature_count = Counter(all_features)

stable_features = [f for f, c in feature_count.items() if c >= 3]

print("Stable features (>=3 parts):")
print(stable_features)
print("Total:", len(stable_features))

Stable features (>=3 parts):
['horizon_25', 'ts_index', 'sub_category_NQ58FVQM', 'sub_code', 'feature_cg', 'sub_category_V8BKY1IV', 'feature_al', 'feature_ce', 'feature_v', 'sub_category_PHHHVYZI', 'feature_cf', 'horizon_10', 'feature_a', 'feature_ar', 'feature_as', 'feature_m', 'feature_aq', 'feature_l', 'feature_ai', 'feature_bg', 'feature_ad', 'feature_i', 'horizon_1', 'feature_am', 'feature_h', 'feature_z', 'horizon_3', 'feature_aj', 'sub_category_PZ9S1Z4V', 'feature_r', 'feature_af', 'feature_n', 'sub_category_DPPUO5X2', 'feature_u', 'feature_y', 'feature_bp', 'feature_ao', 'feature_bz', 'feature_j']
Total: 39
